In [1]:
import mlflow
import mlflow.pyfunc
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import shap
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "../../run")
from const import REPO_PATH
from experiment_config import TRAINGING_CONFIG
sys.path.insert(1, f"{REPO_PATH}")
from src.model.experiment_utils import align_on_keys

/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
features_path = f"{TRAINGING_CONFIG['features_path']}/all_combined_features_2017-24.csv"
seasons = sorted(TRAINGING_CONFIG['seasons'])
target_dfs = [pd.read_csv(f"{TRAINGING_CONFIG['processed_data_path']}/{season}/all_target_df.csv") for season in seasons[1:]]
for df in target_dfs:
    df['date'] = pd.to_datetime(df['date'])
target_df = pd.concat(target_dfs, ignore_index=True)
feature_df = pd.read_csv(features_path)
feature_df['date'] = pd.to_datetime(feature_df['date'])
feature_df, target_df = align_on_keys(feature_df, target_df, TRAINGING_CONFIG['key_columns'])

In [3]:
feature_df.fillna(-1, inplace=True)
numeric_cols = feature_df.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
feature_df[numeric_cols] = scaler.fit_transform(feature_df[numeric_cols])

In [6]:
# Set the model name and load the model from MLflow
model_name = "MLPwithDropoutBN"  # Change if needed
experiment_name = TRAINGING_CONFIG['experiment_name']
tracking_uri = f"{REPO_PATH}/mlflow"
mlflow.set_tracking_uri(tracking_uri)
client = mlflow.tracking.MlflowClient(tracking_uri)
# Search for the run by exact run name (tag 'mlflow.runName')
runs = client.search_runs(
    experiment_ids=[client.get_experiment_by_name(experiment_name).experiment_id],
    filter_string=f"tags.mlflow.runName = 'XBGClassifier_away_goals_n_estimators=200_max_depth=7_learning_rate=0.1_gamma=1'"
)
if len(runs) == 0:
    raise ValueError(f"No runs found for model name: XBGClassifier_away_goals_n_estimators=200_max_depth=7_learning_rate=0.1_gamma=1")
run_id = runs[0].info.run_id
model_uri = f"runs:/{run_id}/model"
print("Run ID:", run_id)
print("Model URI:", model_uri)
artifacts = client.list_artifacts(run_id)
print("Artifacts:", artifacts)
loaded_model = mlflow.pyfunc.load_model(model_uri)

Run ID: e19f3529493049ccbe59bc3e86f3924b
Model URI: runs:/e19f3529493049ccbe59bc3e86f3924b/model
Artifacts: []


TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
# SHAP value analysis

explainer = shap.Explainer(loaded_model.predict, feature_df)
shap_values = explainer(feature_df)
shap.summary_plot(shap_values, feature_df, show=False)
plt.show()